In [1]:
import sys, os

# Add the project root (one level up)
PROJECT_ROOT = os.path.abspath("..")
sys.path.append(PROJECT_ROOT)

print(PROJECT_ROOT)  # sanity check


c:\Users\anaol\repos\langchain\langchain-AI-summit


In [2]:
import re
import uuid
from langsmith.schemas import Run, Example
from langsmith import evaluate, aevaluate, wrappers
from openai import OpenAI
from langsmith import Client
from pydantic import BaseModel
from agent.return_agent import agent
from langchain_core.messages import HumanMessage


# Relevance

In [17]:
def prepare_data(run, example):
    # USER QUESTION (from example input)
    user_msg = example.inputs["messages"][0]["content"]

    # REFERENCE ANSWER (from example output)
    ai_messages = [m for m in example.outputs["messages"] if m["type"] == "ai"]
    reference = ai_messages[-1]["content"] if ai_messages else ""

    # MODEL ANSWER (from run)
    output = run.outputs.get("prediction", "")

    return {
        "question": user_msg,
        "reference": reference,
        "answer": output
    }


In [ ]:
def run_agent(inputs: dict):
    # Extract the user message
    messages = inputs.get("messages", [])
    if not messages:
        raise ValueError("Dataset missing 'messages' list.")

    user_msg = messages[0]["content"]

    thread_id = f"eval-{uuid.uuid4()}"

    # Agent invocation
    result = agent.invoke(
        {"messages": [HumanMessage(content=user_msg)]},
        config={"thread_id": thread_id}
    )

    # Return only final answer
    final_answer = result["messages"][-1].content

    return {"prediction": final_answer}


In [5]:
# Use an LLM-as-a-judge
oai_client = wrappers.wrap_openai(OpenAI())

In [102]:
def relevance(run: Run, example: Example) -> dict:
    """
    Avalia CLAREZA + RELEVÂNCIA da resposta.
    Retorna nota 0..1.
    """
    instructions = """
    Avalie a resposta com base em CLAREZA e RELEVÂNCIA (nota 0 a 1).
    - 1.0 = resposta completa, clara e diretamente relacionada.
    - 0.5 = resposta parcialmente correta.
    - 0.0 = resposta incorreta ou fora do contexto.
    Retorne APENAS a nota, nada mais.
    """

    data = prepare_data(run, example)

    msg = (
        f"Question: {data['question']}\n"
        f"Reference: {data['reference']}\n"
        f"Answer: {data['answer']}"
    )

    resp = oai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": msg}
        ],
    )

    score_str = resp.choices[0].message.content.strip()
    score = float(score_str)

    return {"key": "relevance", "score": score}


In [7]:
client = Client()
client.list_examples(dataset_name="agent-final-output")

<generator object Client.list_examples at 0x000001DE803E6D40>

In [96]:
input_example = {
  "messages": [
    {
      "id": "350c9af6-d946-490c-94c7-26cc536a553a",
      "type": "human",
      "content": "Processe a devolução do pedido e481f51cbdc54678b7cc49136f2d6af7",
      "additional_kwargs": {},
      "response_metadata": {}
    }
  ],
  "decide_path": "general",
  "pdf_context": ""
}

In [ ]:
output_example = {
  "messages": [
    {
      "id": "36975c64-526e-4057-9201-8fa21106878a",
      "type": "human",
      "content": "Há quantos pedidos com status cancelado?",
      "additional_kwargs": {},
      "response_metadata": {}
    },
    {
      "id": "eb2dc206-3d90-46d4-af5d-af92e0c2db87",
      "type": "ai",
      "content": "",
      "tool_calls": [
        {
          "id": "schema_call_1",
          "args": {
            "table_names": "category_translation, customers, geolocation, order_items, order_payments, order_reviews, orders, products, sellers"
          },
          "name": "sql_db_schema",
          "type": "tool_call"
        }
      ],
      "additional_kwargs": {},
      "response_metadata": {},
      "invalid_tool_calls": []
    },
    {
      "id": "eea047e7-e0c8-4a88-b9f8-a3c16b40709c",
      "name": "sql_db_schema",
      "type": "tool",
      "status": "success",
      "content": "\nCREATE TABLE category_translation (\n\tproduct_category_name TEXT, \n\tproduct_category_name_english TEXT\n)\n\n/*\n3 rows from category_translation table:\nproduct_category_name\tproduct_category_name_english\nbeleza_saude\thealth_beauty\ninformatica_acessorios\tcomputers_accessories\nautomotivo\tauto\n*/\n\n\nCREATE TABLE customers (\n\tcustomer_id TEXT, \n\tcustomer_unique_id TEXT, \n\tcustomer_zip_code_prefix INTEGER, \n\tcustomer_city TEXT, \n\tcustomer_state TEXT\n)\n\n/*\n3 rows from customers table:\ncustomer_id\tcustomer_unique_id\tcustomer_zip_code_prefix\tcustomer_city\tcustomer_state\n06b8999e2fba1a1fbc88172c00ba8bc7\t861eff4711a542e4b93843c6dd7febb0\t14409\tfranca\tSP\n18955e83d337fd6b2def6b18a428ac77\t290c77bc529b7ac935b93aa66c333dc3\t9790\tsao bernardo do campo\tSP\n4e7b3e00288586ebd08712fdd0374a03\t060e732b5b29e8181a18229c7b0b2b5e\t1151\tsao paulo\tSP\n*/\n\n\nCREATE TABLE geolocation (\n\tgeolocation_zip_code_prefix INTEGER, \n\tgeolocation_lat REAL, \n\tgeolocation_lng REAL, \n\tgeolocation_city TEXT, \n\tgeolocation_state TEXT\n)\n\n/*\n3 rows from geolocation table:\ngeolocation_zip_code_prefix\tgeolocation_lat\tgeolocation_lng\tgeolocation_city\tgeolocation_state\n1037\t-23.54562128115268\t-46.63929204800168\tsao paulo\tSP\n1046\t-23.54608112703553\t-46.64482029837157\tsao paulo\tSP\n1046\t-23.54612896641469\t-46.64295148361138\tsao paulo\tSP\n*/\n\n\nCREATE TABLE order_items (\n\torder_id TEXT, \n\torder_item_id INTEGER, \n\tproduct_id TEXT, \n\tseller_id TEXT, \n\tshipping_limit_date TEXT, \n\tprice REAL, \n\tfreight_value REAL\n)\n\n/*\n3 rows from order_items table:\norder_id\torder_item_id\tproduct_id\tseller_id\tshipping_limit_date\tprice\tfreight_value\n00010242fe8c5a6d1ba2dd792cb16214\t1\t4244733e06e7ecb4970a6e2683c13e61\t48436dade18ac8b2bce089ec2a041202\t2017-09-19 09:45:35\t58.9\t13.29\n00018f77f2f0320c557190d7a144bdd3\t1\te5f2d52b802189ee658865ca93d83a8f\tdd7ddc04e1b6c2c614352b383efe2d36\t2017-05-03 11:05:13\t239.9\t19.93\n000229ec398224ef6ca0657da4fc703e\t1\tc777355d18b72b67abbeef9df44fd0fd\t5b51032eddd242adc84c38acab88f23d\t2018-01-18 14:48:30\t199.0\t17.87\n*/\n\n\nCREATE TABLE order_payments (\n\torder_id TEXT, \n\tpayment_sequential INTEGER, \n\tpayment_type TEXT, \n\tpayment_installments INTEGER, \n\tpayment_value REAL\n)\n\n/*\n3 rows from order_payments table:\norder_id\tpayment_sequential\tpayment_type\tpayment_installments\tpayment_value\nb81ef226f3fe1789b1e8b2acac839d17\t1\tcredit_card\t8\t99.33\na9810da82917af2d9aefd1278f1dcfa0\t1\tcredit_card\t1\t24.39\n25e8ea4e93396b6fa0d3dd708e76c1bd\t1\tcredit_card\t1\t65.71\n*/\n\n\nCREATE TABLE order_reviews (\n\treview_id TEXT, \n\torder_id TEXT, \n\treview_score INTEGER, \n\treview_comment_title TEXT, \n\treview_comment_message TEXT, \n\treview_creation_date TEXT, \n\treview_answer_timestamp TEXT\n)\n\n/*\n3 rows from order_reviews table:\nreview_id\torder_id\treview_score\treview_comment_title\treview_comment_message\treview_creation_date\treview_answer_timestamp\n7bc2406110b926393aa56f80a40eba40\t73fc7af87114b39712e6da79b0a377eb\t4\tNone\tNone\t2018-01-18 00:00:00\t2018-01-18 21:46:59\n80e641a11e56f04c1ad469d5645fdfde\ta548910a1c6147796b98fdf73dbeba33\t5\tNone\tNone\t2018-03-10 00:00:00\t2018-03-11 03:05:13\n228ce5500dc1d8e020d8d1322874b6f0\tf9e4b658b201a9f2ecdecbb34bed034b\t5\tNone\tNone\t2018-02-17 00:00:00\t2018-02-18 14:36:24\n*/\n\n\nCREATE TABLE orders (\n\torder_id TEXT, \n\tcustomer_id TEXT, \n\torder_status TEXT, \n\torder_purchase_timestamp TEXT, \n\torder_approved_at TEXT, \n\torder_delivered_carrier_date TEXT, \n\torder_delivered_customer_date TEXT, \n\torder_estimated_delivery_date TEXT\n)\n\n/*\n3 rows from orders table:\norder_id\tcustomer_id\torder_status\torder_purchase_timestamp\torder_approved_at\torder_delivered_carrier_date\torder_delivered_customer_date\torder_estimated_delivery_date\ne481f51cbdc54678b7cc49136f2d6af7\t9ef432eb6251297304e76186b10a928d\treturn_requested\t2024-10-02 10:56:33\t2024-10-02 11:07:15\t2024-10-04 19:55:00\t2024-10-18 21:25:13\t2024-10-18 00:00:00\n53cdb2fc8bc7dce0b6741e2150273451\tb0830fb4747a6c6d20dea0b8c802d7ef\treturn_requested\t2025-07-24 20:41:37\t2025-07-26 03:24:27\t2025-07-26 14:31:00\t2025-08-07 15:27:45\t2025-08-13 00:00:00\n47770eb9100c2d0c44946d9cf07ec65d\t41ce2a54c0b03bf3443c3d931a367089\tdelivered\t2025-08-08 08:38:49\t2025-08-08 08:55:23\t2025-08-08 13:50:00\t2025-08-17 18:06:29\t2025-09-04 00:00:00\n*/\n\n\nCREATE TABLE products (\n\tproduct_id TEXT, \n\tproduct_category_name TEXT, \n\tproduct_name_lenght REAL, \n\tproduct_description_lenght REAL, \n\tproduct_photos_qty REAL, \n\tproduct_weight_g REAL, \n\tproduct_length_cm REAL, \n\tproduct_height_cm REAL, \n\tproduct_width_cm REAL\n)\n\n/*\n3 rows from products table:\nproduct_id\tproduct_category_name\tproduct_name_lenght\tproduct_description_lenght\tproduct_photos_qty\tproduct_weight_g\tproduct_length_cm\tproduct_height_cm\tproduct_width_cm\n1e9e8ef04dbcff4541ed26657ea517e5\tperfumaria\t40.0\t287.0\t1.0\t225.0\t16.0\t10.0\t14.0\n3aa071139cb16b67ca9e5dea641aaa2f\tartes\t44.0\t276.0\t1.0\t1000.0\t30.0\t18.0\t20.0\n96bd76ec8810374ed1b65e291975717f\tesporte_lazer\t46.0\t250.0\t1.0\t154.0\t18.0\t9.0\t15.0\n*/\n\n\nCREATE TABLE sellers (\n\tseller_id TEXT, \n\tseller_zip_code_prefix INTEGER, \n\tseller_city TEXT, \n\tseller_state TEXT\n)\n\n/*\n3 rows from sellers table:\nseller_id\tseller_zip_code_prefix\tseller_city\tseller_state\n3442f8959a84dea7ee197c632cb2df15\t13023\tcampinas\tSP\nd1b65fc7debc3361ea86b5f14c68d2e2\t13844\tmogi guacu\tSP\nce3ad9de960102d0677a81f5d0bb7b2d\t20031\trio de janeiro\tRJ\n*/",
      "tool_call_id": "schema_call_1",
      "additional_kwargs": {},
      "response_metadata": {}
    },
    {
      "id": "lc_run--5a7814a0-f370-4f6b-b981-a4951f750511-0",
      "type": "ai",
      "content": "",
      "tool_calls": [
        {
          "id": "call_yWcoT5uHhrK57AJr4ZXRG6Mg",
          "args": {
            "query": "SELECT COUNT(*) AS canceled_orders FROM orders WHERE order_status = 'canceled';"
          },
          "name": "sql_db_query",
          "type": "tool_call"
        }
      ],
      "usage_metadata": {
        "input_tokens": 2409,
        "total_tokens": 2640,
        "output_tokens": 231,
        "input_token_details": {
          "audio": 0,
          "cache_read": 0
        },
        "output_token_details": {
          "audio": 0,
          "reasoning": 192
        }
      },
      "additional_kwargs": {
        "refusal": null
      },
      "response_metadata": {
        "id": "chatcmpl-CdG1wqGMyGoEjzjnzcRTVLMkk0ieT",
        "logprobs": null,
        "model_name": "gpt-5-2025-08-07",
        "token_usage": {
          "total_tokens": 2640,
          "prompt_tokens": 2409,
          "completion_tokens": 231,
          "prompt_tokens_details": {
            "audio_tokens": 0,
            "cached_tokens": 0
          },
          "completion_tokens_details": {
            "audio_tokens": 0,
            "reasoning_tokens": 192,
            "accepted_prediction_tokens": 0,
            "rejected_prediction_tokens": 0
          }
        },
        "service_tier": "default",
        "finish_reason": "tool_calls",
        "model_provider": "openai",
        "system_fingerprint": null
      },
      "invalid_tool_calls": []
    },
    {
      "id": "e5c51a86-a971-4eca-b14d-4cbfb14ab98d",
      "name": "sql_db_query",
      "type": "tool",
      "status": "success",
      "content": "[(625,)]",
      "tool_call_id": "call_yWcoT5uHhrK57AJr4ZXRG6Mg",
      "additional_kwargs": {},
      "response_metadata": {}
    },
    {
      "id": "lc_run--66fa8e42-3a80-4546-86d8-9bd3afffef0a-0",
      "type": "ai",
      "content": "Há 625 pedidos com status cancelado.",
      "tool_calls": [],
      "usage_metadata": {
        "input_tokens": 631,
        "total_tokens": 649,
        "output_tokens": 18,
        "input_token_details": {
          "audio": 0,
          "cache_read": 0
        },
        "output_token_details": {
          "audio": 0,
          "reasoning": 0
        }
      },
      "additional_kwargs": {
        "refusal": null
      },
      "response_metadata": {
        "id": "chatcmpl-CdG208JLG6PYkM26MEnhBqUl5KNQp",
        "logprobs": null,
        "model_name": "gpt-5-2025-08-07",
        "token_usage": {
          "total_tokens": 649,
          "prompt_tokens": 631,
          "completion_tokens": 18,
          "prompt_tokens_details": {
            "audio_tokens": 0,
            "cached_tokens": 0
          },
          "completion_tokens_details": {
            "audio_tokens": 0,
            "reasoning_tokens": 0,
            "accepted_prediction_tokens": 0,
            "rejected_prediction_tokens": 0
          }
        },
        "service_tier": "default",
        "finish_reason": "stop",
        "model_provider": "openai",
        "system_fingerprint": null
      },
      "invalid_tool_calls": []
    },
    {
      "id": "a6dc106c-cc1c-4121-8755-395a94ef56ae",
      "type": "human",
      "content": "Quais 3 vendedores que tiveram maior número de entregas atrasadas em 2024?",
      "additional_kwargs": {},
      "response_metadata": {}
    },
    {
      "id": "b8681130-950b-4b89-97e4-2ee2336e3cf2",
      "type": "ai",
      "content": "",
      "tool_calls": [
        {
          "id": "schema_call_7",
          "args": {
            "table_names": "category_translation, customers, geolocation, order_items, order_payments, order_reviews, orders, products, sellers"
          },
          "name": "sql_db_schema",
          "type": "tool_call"
        }
      ],
      "additional_kwargs": {},
      "response_metadata": {},
      "invalid_tool_calls": []
    },
    {
      "id": "fa8c684d-1f50-4abb-be5b-01b3fa22b1fe",
      "name": "sql_db_schema",
      "type": "tool",
      "status": "success",
      "content": "\nCREATE TABLE category_translation (\n\tproduct_category_name TEXT, \n\tproduct_category_name_english TEXT\n)\n\n/*\n3 rows from category_translation table:\nproduct_category_name\tproduct_category_name_english\nbeleza_saude\thealth_beauty\ninformatica_acessorios\tcomputers_accessories\nautomotivo\tauto\n*/\n\n\nCREATE TABLE customers (\n\tcustomer_id TEXT, \n\tcustomer_unique_id TEXT, \n\tcustomer_zip_code_prefix INTEGER, \n\tcustomer_city TEXT, \n\tcustomer_state TEXT\n)\n\n/*\n3 rows from customers table:\ncustomer_id\tcustomer_unique_id\tcustomer_zip_code_prefix\tcustomer_city\tcustomer_state\n06b8999e2fba1a1fbc88172c00ba8bc7\t861eff4711a542e4b93843c6dd7febb0\t14409\tfranca\tSP\n18955e83d337fd6b2def6b18a428ac77\t290c77bc529b7ac935b93aa66c333dc3\t9790\tsao bernardo do campo\tSP\n4e7b3e00288586ebd08712fdd0374a03\t060e732b5b29e8181a18229c7b0b2b5e\t1151\tsao paulo\tSP\n*/\n\n\nCREATE TABLE geolocation (\n\tgeolocation_zip_code_prefix INTEGER, \n\tgeolocation_lat REAL, \n\tgeolocation_lng REAL, \n\tgeolocation_city TEXT, \n\tgeolocation_state TEXT\n)\n\n/*\n3 rows from geolocation table:\ngeolocation_zip_code_prefix\tgeolocation_lat\tgeolocation_lng\tgeolocation_city\tgeolocation_state\n1037\t-23.54562128115268\t-46.63929204800168\tsao paulo\tSP\n1046\t-23.54608112703553\t-46.64482029837157\tsao paulo\tSP\n1046\t-23.54612896641469\t-46.64295148361138\tsao paulo\tSP\n*/\n\n\nCREATE TABLE order_items (\n\torder_id TEXT, \n\torder_item_id INTEGER, \n\tproduct_id TEXT, \n\tseller_id TEXT, \n\tshipping_limit_date TEXT, \n\tprice REAL, \n\tfreight_value REAL\n)\n\n/*\n3 rows from order_items table:\norder_id\torder_item_id\tproduct_id\tseller_id\tshipping_limit_date\tprice\tfreight_value\n00010242fe8c5a6d1ba2dd792cb16214\t1\t4244733e06e7ecb4970a6e2683c13e61\t48436dade18ac8b2bce089ec2a041202\t2017-09-19 09:45:35\t58.9\t13.29\n00018f77f2f0320c557190d7a144bdd3\t1\te5f2d52b802189ee658865ca93d83a8f\tdd7ddc04e1b6c2c614352b383efe2d36\t2017-05-03 11:05:13\t239.9\t19.93\n000229ec398224ef6ca0657da4fc703e\t1\tc777355d18b72b67abbeef9df44fd0fd\t5b51032eddd242adc84c38acab88f23d\t2018-01-18 14:48:30\t199.0\t17.87\n*/\n\n\nCREATE TABLE order_payments (\n\torder_id TEXT, \n\tpayment_sequential INTEGER, \n\tpayment_type TEXT, \n\tpayment_installments INTEGER, \n\tpayment_value REAL\n)\n\n/*\n3 rows from order_payments table:\norder_id\tpayment_sequential\tpayment_type\tpayment_installments\tpayment_value\nb81ef226f3fe1789b1e8b2acac839d17\t1\tcredit_card\t8\t99.33\na9810da82917af2d9aefd1278f1dcfa0\t1\tcredit_card\t1\t24.39\n25e8ea4e93396b6fa0d3dd708e76c1bd\t1\tcredit_card\t1\t65.71\n*/\n\n\nCREATE TABLE order_reviews (\n\treview_id TEXT, \n\torder_id TEXT, \n\treview_score INTEGER, \n\treview_comment_title TEXT, \n\treview_comment_message TEXT, \n\treview_creation_date TEXT, \n\treview_answer_timestamp TEXT\n)\n\n/*\n3 rows from order_reviews table:\nreview_id\torder_id\treview_score\treview_comment_title\treview_comment_message\treview_creation_date\treview_answer_timestamp\n7bc2406110b926393aa56f80a40eba40\t73fc7af87114b39712e6da79b0a377eb\t4\tNone\tNone\t2018-01-18 00:00:00\t2018-01-18 21:46:59\n80e641a11e56f04c1ad469d5645fdfde\ta548910a1c6147796b98fdf73dbeba33\t5\tNone\tNone\t2018-03-10 00:00:00\t2018-03-11 03:05:13\n228ce5500dc1d8e020d8d1322874b6f0\tf9e4b658b201a9f2ecdecbb34bed034b\t5\tNone\tNone\t2018-02-17 00:00:00\t2018-02-18 14:36:24\n*/\n\n\nCREATE TABLE orders (\n\torder_id TEXT, \n\tcustomer_id TEXT, \n\torder_status TEXT, \n\torder_purchase_timestamp TEXT, \n\torder_approved_at TEXT, \n\torder_delivered_carrier_date TEXT, \n\torder_delivered_customer_date TEXT, \n\torder_estimated_delivery_date TEXT\n)\n\n/*\n3 rows from orders table:\norder_id\tcustomer_id\torder_status\torder_purchase_timestamp\torder_approved_at\torder_delivered_carrier_date\torder_delivered_customer_date\torder_estimated_delivery_date\ne481f51cbdc54678b7cc49136f2d6af7\t9ef432eb6251297304e76186b10a928d\treturn_requested\t2024-10-02 10:56:33\t2024-10-02 11:07:15\t2024-10-04 19:55:00\t2024-10-18 21:25:13\t2024-10-18 00:00:00\n53cdb2fc8bc7dce0b6741e2150273451\tb0830fb4747a6c6d20dea0b8c802d7ef\treturn_requested\t2025-07-24 20:41:37\t2025-07-26 03:24:27\t2025-07-26 14:31:00\t2025-08-07 15:27:45\t2025-08-13 00:00:00\n47770eb9100c2d0c44946d9cf07ec65d\t41ce2a54c0b03bf3443c3d931a367089\tdelivered\t2025-08-08 08:38:49\t2025-08-08 08:55:23\t2025-08-08 13:50:00\t2025-08-17 18:06:29\t2025-09-04 00:00:00\n*/\n\n\nCREATE TABLE products (\n\tproduct_id TEXT, \n\tproduct_category_name TEXT, \n\tproduct_name_lenght REAL, \n\tproduct_description_lenght REAL, \n\tproduct_photos_qty REAL, \n\tproduct_weight_g REAL, \n\tproduct_length_cm REAL, \n\tproduct_height_cm REAL, \n\tproduct_width_cm REAL\n)\n\n/*\n3 rows from products table:\nproduct_id\tproduct_category_name\tproduct_name_lenght\tproduct_description_lenght\tproduct_photos_qty\tproduct_weight_g\tproduct_length_cm\tproduct_height_cm\tproduct_width_cm\n1e9e8ef04dbcff4541ed26657ea517e5\tperfumaria\t40.0\t287.0\t1.0\t225.0\t16.0\t10.0\t14.0\n3aa071139cb16b67ca9e5dea641aaa2f\tartes\t44.0\t276.0\t1.0\t1000.0\t30.0\t18.0\t20.0\n96bd76ec8810374ed1b65e291975717f\tesporte_lazer\t46.0\t250.0\t1.0\t154.0\t18.0\t9.0\t15.0\n*/\n\n\nCREATE TABLE sellers (\n\tseller_id TEXT, \n\tseller_zip_code_prefix INTEGER, \n\tseller_city TEXT, \n\tseller_state TEXT\n)\n\n/*\n3 rows from sellers table:\nseller_id\tseller_zip_code_prefix\tseller_city\tseller_state\n3442f8959a84dea7ee197c632cb2df15\t13023\tcampinas\tSP\nd1b65fc7debc3361ea86b5f14c68d2e2\t13844\tmogi guacu\tSP\nce3ad9de960102d0677a81f5d0bb7b2d\t20031\trio de janeiro\tRJ\n*/",
      "tool_call_id": "schema_call_7",
      "additional_kwargs": {},
      "response_metadata": {}
    },
    {
      "id": "lc_run--ed42634c-f6f0-41ec-806b-1d9eb6d8b240-0",
      "type": "ai",
      "content": "",
      "tool_calls": [
        {
          "id": "call_bnPoMT416ZzlNotbcFZKww0Y",
          "args": {
            "query": "SELECT oi.seller_id, s.seller_city, s.seller_state, COUNT(DISTINCT oi.order_id) AS late_deliveries\nFROM order_items oi\nJOIN orders o ON o.order_id = oi.order_id\nJOIN sellers s ON s.seller_id = oi.seller_id\nWHERE o.order_delivered_customer_date IS NOT NULL\n  AND DATE(o.order_delivered_customer_date) BETWEEN '2024-01-01' AND '2024-12-31'\n  AND DATE(o.order_delivered_customer_date) > DATE(o.order_estimated_delivery_date)\nGROUP BY oi.seller_id, s.seller_city, s.seller_state\nORDER BY late_deliveries DESC\nLIMIT 3;"
          },
          "name": "sql_db_query",
          "type": "tool_call"
        }
      ],
      "usage_metadata": {
        "input_tokens": 4540,
        "total_tokens": 5673,
        "output_tokens": 1133,
        "input_token_details": {
          "audio": 0,
          "cache_read": 2304
        },
        "output_token_details": {
          "audio": 0,
          "reasoning": 960
        }
      },
      "additional_kwargs": {
        "refusal": null
      },
      "response_metadata": {
        "id": "chatcmpl-CdG32XZyLCKsQSxf27GNzZETn8yKT",
        "logprobs": null,
        "model_name": "gpt-5-2025-08-07",
        "token_usage": {
          "total_tokens": 5673,
          "prompt_tokens": 4540,
          "completion_tokens": 1133,
          "prompt_tokens_details": {
            "audio_tokens": 0,
            "cached_tokens": 2304
          },
          "completion_tokens_details": {
            "audio_tokens": 0,
            "reasoning_tokens": 960,
            "accepted_prediction_tokens": 0,
            "rejected_prediction_tokens": 0
          }
        },
        "service_tier": "default",
        "finish_reason": "tool_calls",
        "model_provider": "openai",
        "system_fingerprint": null
      },
      "invalid_tool_calls": []
    },
    {
      "id": "53f730e5-2cc3-42e5-b93a-3d91d7464ea6",
      "name": "sql_db_query",
      "type": "tool",
      "status": "success",
      "content": "[('4a3ca9315b744ce9f8e9374361493884', 'ibitinga', 'SP', 65), ('1f50f920176fa81dab994f9023523100', 'sao jose do rio preto', 'SP', 42), ('7c67e1448b00f6e969d365cea6b010ab', 'itaquaquecetuba', 'SP', 40)]",
      "tool_call_id": "call_bnPoMT416ZzlNotbcFZKww0Y",
      "additional_kwargs": {},
      "response_metadata": {}
    },
    {
      "id": "lc_run--60719cde-6428-4432-8d7b-6e590f36ba92-0",
      "type": "ai",
      "content": "Top 3 vendedores com mais entregas atrasadas em 2024:\n1. 4a3ca9315b744ce9f8e9374361493884 (Ibitinga, SP): 65 entregas atrasadas\n2. 1f50f920176fa81dab994f9023523100 (São José do Rio Preto, SP): 42\n3. 7c67e1448b00f6e969d365cea6b010ab (Itaquaquecetuba, SP): 40",
      "tool_calls": [],
      "usage_metadata": {
        "input_tokens": 868,
        "total_tokens": 991,
        "output_tokens": 123,
        "input_token_details": {
          "audio": 0,
          "cache_read": 0
        },
        "output_token_details": {
          "audio": 0,
          "reasoning": 0
        }
      },
      "additional_kwargs": {
        "refusal": null
      },
      "response_metadata": {
        "id": "chatcmpl-CdG3E3zgzzFcMwl4rKVEeNFCmL2D0",
        "logprobs": null,
        "model_name": "gpt-5-2025-08-07",
        "token_usage": {
          "total_tokens": 991,
          "prompt_tokens": 868,
          "completion_tokens": 123,
          "prompt_tokens_details": {
            "audio_tokens": 0,
            "cached_tokens": 0
          },
          "completion_tokens_details": {
            "audio_tokens": 0,
            "reasoning_tokens": 0,
            "accepted_prediction_tokens": 0,
            "rejected_prediction_tokens": 0
          }
        },
        "service_tier": "default",
        "finish_reason": "stop",
        "model_provider": "openai",
        "system_fingerprint": null
      },
      "invalid_tool_calls": []
    },
    {
      "id": "83eccb49-8882-4eec-ab47-ec33c5e297a1",
      "type": "human",
      "content": "No caso de devolução quem arca com o custo do envio?",
      "additional_kwargs": {},
      "response_metadata": {}
    },
    {
      "id": "lc_run--11962e3e-d54d-4927-90e0-46854dff2ab8-0",
      "type": "ai",
      "content": "Depende do motivo:\n\n- Erro do vendedor (defeituoso, errado, incompleto): o vendedor paga o frete de envio e de devolução.\n- Arrependimento do comprador: a BIX paga o frete de devolução.\n- Disputa: a BIX decide quem arca com os custos após avaliação.",
      "tool_calls": [],
      "usage_metadata": {
        "input_tokens": 1972,
        "total_tokens": 2115,
        "output_tokens": 143,
        "input_token_details": {
          "audio": 0,
          "cache_read": 0
        },
        "output_token_details": {
          "audio": 0,
          "reasoning": 64
        }
      },
      "additional_kwargs": {
        "refusal": null
      },
      "response_metadata": {
        "id": "chatcmpl-CdG6LKBxBE3lGGkd7cosUMDZDSzhV",
        "logprobs": null,
        "model_name": "gpt-5-2025-08-07",
        "token_usage": {
          "total_tokens": 2115,
          "prompt_tokens": 1972,
          "completion_tokens": 143,
          "prompt_tokens_details": {
            "audio_tokens": 0,
            "cached_tokens": 0
          },
          "completion_tokens_details": {
            "audio_tokens": 0,
            "reasoning_tokens": 64,
            "accepted_prediction_tokens": 0,
            "rejected_prediction_tokens": 0
          }
        },
        "service_tier": "default",
        "finish_reason": "stop",
        "model_provider": "openai",
        "system_fingerprint": null
      },
      "invalid_tool_calls": []
    },
    {
      "id": "355e83d9-fc51-4393-95dd-6587bcc57d92",
      "type": "human",
      "content": "Quantos pedidos entregues estão dentro do prazo de devolução por arrependimento?",
      "additional_kwargs": {},
      "response_metadata": {}
    },
    {
      "id": "26b758d0-7815-4ccf-a7e5-5f39ae52947e",
      "type": "ai",
      "content": "",
      "tool_calls": [
        {
          "id": "schema_call_15",
          "args": {
            "table_names": "category_translation, customers, geolocation, order_items, order_payments, order_reviews, orders, products, sellers"
          },
          "name": "sql_db_schema",
          "type": "tool_call"
        }
      ],
      "additional_kwargs": {},
      "response_metadata": {},
      "invalid_tool_calls": []
    },
    {
      "id": "7d007d8f-dcf8-4288-8679-be7b6c306000",
      "name": "sql_db_schema",
      "type": "tool",
      "status": "success",
      "content": "\nCREATE TABLE category_translation (\n\tproduct_category_name TEXT, \n\tproduct_category_name_english TEXT\n)\n\n/*\n3 rows from category_translation table:\nproduct_category_name\tproduct_category_name_english\nbeleza_saude\thealth_beauty\ninformatica_acessorios\tcomputers_accessories\nautomotivo\tauto\n*/\n\n\nCREATE TABLE customers (\n\tcustomer_id TEXT, \n\tcustomer_unique_id TEXT, \n\tcustomer_zip_code_prefix INTEGER, \n\tcustomer_city TEXT, \n\tcustomer_state TEXT\n)\n\n/*\n3 rows from customers table:\ncustomer_id\tcustomer_unique_id\tcustomer_zip_code_prefix\tcustomer_city\tcustomer_state\n06b8999e2fba1a1fbc88172c00ba8bc7\t861eff4711a542e4b93843c6dd7febb0\t14409\tfranca\tSP\n18955e83d337fd6b2def6b18a428ac77\t290c77bc529b7ac935b93aa66c333dc3\t9790\tsao bernardo do campo\tSP\n4e7b3e00288586ebd08712fdd0374a03\t060e732b5b29e8181a18229c7b0b2b5e\t1151\tsao paulo\tSP\n*/\n\n\nCREATE TABLE geolocation (\n\tgeolocation_zip_code_prefix INTEGER, \n\tgeolocation_lat REAL, \n\tgeolocation_lng REAL, \n\tgeolocation_city TEXT, \n\tgeolocation_state TEXT\n)\n\n/*\n3 rows from geolocation table:\ngeolocation_zip_code_prefix\tgeolocation_lat\tgeolocation_lng\tgeolocation_city\tgeolocation_state\n1037\t-23.54562128115268\t-46.63929204800168\tsao paulo\tSP\n1046\t-23.54608112703553\t-46.64482029837157\tsao paulo\tSP\n1046\t-23.54612896641469\t-46.64295148361138\tsao paulo\tSP\n*/\n\n\nCREATE TABLE order_items (\n\torder_id TEXT, \n\torder_item_id INTEGER, \n\tproduct_id TEXT, \n\tseller_id TEXT, \n\tshipping_limit_date TEXT, \n\tprice REAL, \n\tfreight_value REAL\n)\n\n/*\n3 rows from order_items table:\norder_id\torder_item_id\tproduct_id\tseller_id\tshipping_limit_date\tprice\tfreight_value\n00010242fe8c5a6d1ba2dd792cb16214\t1\t4244733e06e7ecb4970a6e2683c13e61\t48436dade18ac8b2bce089ec2a041202\t2017-09-19 09:45:35\t58.9\t13.29\n00018f77f2f0320c557190d7a144bdd3\t1\te5f2d52b802189ee658865ca93d83a8f\tdd7ddc04e1b6c2c614352b383efe2d36\t2017-05-03 11:05:13\t239.9\t19.93\n000229ec398224ef6ca0657da4fc703e\t1\tc777355d18b72b67abbeef9df44fd0fd\t5b51032eddd242adc84c38acab88f23d\t2018-01-18 14:48:30\t199.0\t17.87\n*/\n\n\nCREATE TABLE order_payments (\n\torder_id TEXT, \n\tpayment_sequential INTEGER, \n\tpayment_type TEXT, \n\tpayment_installments INTEGER, \n\tpayment_value REAL\n)\n\n/*\n3 rows from order_payments table:\norder_id\tpayment_sequential\tpayment_type\tpayment_installments\tpayment_value\nb81ef226f3fe1789b1e8b2acac839d17\t1\tcredit_card\t8\t99.33\na9810da82917af2d9aefd1278f1dcfa0\t1\tcredit_card\t1\t24.39\n25e8ea4e93396b6fa0d3dd708e76c1bd\t1\tcredit_card\t1\t65.71\n*/\n\n\nCREATE TABLE order_reviews (\n\treview_id TEXT, \n\torder_id TEXT, \n\treview_score INTEGER, \n\treview_comment_title TEXT, \n\treview_comment_message TEXT, \n\treview_creation_date TEXT, \n\treview_answer_timestamp TEXT\n)\n\n/*\n3 rows from order_reviews table:\nreview_id\torder_id\treview_score\treview_comment_title\treview_comment_message\treview_creation_date\treview_answer_timestamp\n7bc2406110b926393aa56f80a40eba40\t73fc7af87114b39712e6da79b0a377eb\t4\tNone\tNone\t2018-01-18 00:00:00\t2018-01-18 21:46:59\n80e641a11e56f04c1ad469d5645fdfde\ta548910a1c6147796b98fdf73dbeba33\t5\tNone\tNone\t2018-03-10 00:00:00\t2018-03-11 03:05:13\n228ce5500dc1d8e020d8d1322874b6f0\tf9e4b658b201a9f2ecdecbb34bed034b\t5\tNone\tNone\t2018-02-17 00:00:00\t2018-02-18 14:36:24\n*/\n\n\nCREATE TABLE orders (\n\torder_id TEXT, \n\tcustomer_id TEXT, \n\torder_status TEXT, \n\torder_purchase_timestamp TEXT, \n\torder_approved_at TEXT, \n\torder_delivered_carrier_date TEXT, \n\torder_delivered_customer_date TEXT, \n\torder_estimated_delivery_date TEXT\n)\n\n/*\n3 rows from orders table:\norder_id\tcustomer_id\torder_status\torder_purchase_timestamp\torder_approved_at\torder_delivered_carrier_date\torder_delivered_customer_date\torder_estimated_delivery_date\ne481f51cbdc54678b7cc49136f2d6af7\t9ef432eb6251297304e76186b10a928d\treturn_requested\t2024-10-02 10:56:33\t2024-10-02 11:07:15\t2024-10-04 19:55:00\t2024-10-18 21:25:13\t2024-10-18 00:00:00\n53cdb2fc8bc7dce0b6741e2150273451\tb0830fb4747a6c6d20dea0b8c802d7ef\treturn_requested\t2025-07-24 20:41:37\t2025-07-26 03:24:27\t2025-07-26 14:31:00\t2025-08-07 15:27:45\t2025-08-13 00:00:00\n47770eb9100c2d0c44946d9cf07ec65d\t41ce2a54c0b03bf3443c3d931a367089\tdelivered\t2025-08-08 08:38:49\t2025-08-08 08:55:23\t2025-08-08 13:50:00\t2025-08-17 18:06:29\t2025-09-04 00:00:00\n*/\n\n\nCREATE TABLE products (\n\tproduct_id TEXT, \n\tproduct_category_name TEXT, \n\tproduct_name_lenght REAL, \n\tproduct_description_lenght REAL, \n\tproduct_photos_qty REAL, \n\tproduct_weight_g REAL, \n\tproduct_length_cm REAL, \n\tproduct_height_cm REAL, \n\tproduct_width_cm REAL\n)\n\n/*\n3 rows from products table:\nproduct_id\tproduct_category_name\tproduct_name_lenght\tproduct_description_lenght\tproduct_photos_qty\tproduct_weight_g\tproduct_length_cm\tproduct_height_cm\tproduct_width_cm\n1e9e8ef04dbcff4541ed26657ea517e5\tperfumaria\t40.0\t287.0\t1.0\t225.0\t16.0\t10.0\t14.0\n3aa071139cb16b67ca9e5dea641aaa2f\tartes\t44.0\t276.0\t1.0\t1000.0\t30.0\t18.0\t20.0\n96bd76ec8810374ed1b65e291975717f\tesporte_lazer\t46.0\t250.0\t1.0\t154.0\t18.0\t9.0\t15.0\n*/\n\n\nCREATE TABLE sellers (\n\tseller_id TEXT, \n\tseller_zip_code_prefix INTEGER, \n\tseller_city TEXT, \n\tseller_state TEXT\n)\n\n/*\n3 rows from sellers table:\nseller_id\tseller_zip_code_prefix\tseller_city\tseller_state\n3442f8959a84dea7ee197c632cb2df15\t13023\tcampinas\tSP\nd1b65fc7debc3361ea86b5f14c68d2e2\t13844\tmogi guacu\tSP\nce3ad9de960102d0677a81f5d0bb7b2d\t20031\trio de janeiro\tRJ\n*/",
      "tool_call_id": "schema_call_15",
      "additional_kwargs": {},
      "response_metadata": {}
    },
    {
      "id": "lc_run--d67a25fa-0f46-4be5-ba8c-020281f095d8-0",
      "type": "ai",
      "content": "",
      "tool_calls": [
        {
          "id": "call_yUCg2QZVxGdJgZ1hugrd0llS",
          "args": {
            "query": "SELECT COUNT(*) AS pedidos_dentro_prazo_7d\nFROM orders\nWHERE order_status = 'delivered'\n  AND order_delivered_customer_date IS NOT NULL\n  AND DATE(order_delivered_customer_date) >= DATE('now', '-7 days');"
          },
          "name": "sql_db_query",
          "type": "tool_call"
        }
      ],
      "usage_metadata": {
        "input_tokens": 7095,
        "total_tokens": 8005,
        "output_tokens": 910,
        "input_token_details": {
          "audio": 0,
          "cache_read": 4480
        },
        "output_token_details": {
          "audio": 0,
          "reasoning": 832
        }
      },
      "additional_kwargs": {
        "refusal": null
      },
      "response_metadata": {
        "id": "chatcmpl-CdG7wBlBup2VflPJ6Qzo5EzIfJSow",
        "logprobs": null,
        "model_name": "gpt-5-2025-08-07",
        "token_usage": {
          "total_tokens": 8005,
          "prompt_tokens": 7095,
          "completion_tokens": 910,
          "prompt_tokens_details": {
            "audio_tokens": 0,
            "cached_tokens": 4480
          },
          "completion_tokens_details": {
            "audio_tokens": 0,
            "reasoning_tokens": 832,
            "accepted_prediction_tokens": 0,
            "rejected_prediction_tokens": 0
          }
        },
        "service_tier": "default",
        "finish_reason": "tool_calls",
        "model_provider": "openai",
        "system_fingerprint": null
      },
      "invalid_tool_calls": []
    },
    {
      "id": "0f0b24c9-d698-4a44-afa8-4e6a8936a1a9",
      "name": "sql_db_query",
      "type": "tool",
      "status": "success",
      "content": "[(8,)]",
      "tool_call_id": "call_yUCg2QZVxGdJgZ1hugrd0llS",
      "additional_kwargs": {},
      "response_metadata": {}
    },
    {
      "id": "lc_run--bbcc1ebb-1794-45a1-85b6-0b6df3b56738-0",
      "type": "ai",
      "content": "8 pedidos. Conforme a política, arrependimento é elegível até 7 dias após o recebimento, e temos 8 pedidos entregues nesse intervalo.",
      "tool_calls": [],
      "usage_metadata": {
        "input_tokens": 1943,
        "total_tokens": 2048,
        "output_tokens": 105,
        "input_token_details": {
          "audio": 0,
          "cache_read": 1792
        },
        "output_token_details": {
          "audio": 0,
          "reasoning": 64
        }
      },
      "additional_kwargs": {
        "refusal": null
      },
      "response_metadata": {
        "id": "chatcmpl-CdG8BjEALFeFk7IpHt8FKjusYz7dh",
        "logprobs": null,
        "model_name": "gpt-5-2025-08-07",
        "token_usage": {
          "total_tokens": 2048,
          "prompt_tokens": 1943,
          "completion_tokens": 105,
          "prompt_tokens_details": {
            "audio_tokens": 0,
            "cached_tokens": 1792
          },
          "completion_tokens_details": {
            "audio_tokens": 0,
            "reasoning_tokens": 64,
            "accepted_prediction_tokens": 0,
            "rejected_prediction_tokens": 0
          }
        },
        "service_tier": "default",
        "finish_reason": "stop",
        "model_provider": "openai",
        "system_fingerprint": null
      },
      "invalid_tool_calls": []
    },
    {
      "id": "53cc108c-42a8-43a0-9ae7-acda1a2614c6",
      "type": "human",
      "content": "O vendedor 3442f8959a84dea7ee197c632cb2df15 é confiável?",
      "additional_kwargs": {},
      "response_metadata": {}
    },
    {
      "id": "c04f720f-5e97-4442-8305-d416096d4c23",
      "type": "ai",
      "content": "",
      "tool_calls": [
        {
          "id": "seller_reliability_call_21",
          "args": {
            "seller_id": "3442f8959a84dea7ee197c632cb2df15"
          },
          "name": "analyze_seller_reliability",
          "type": "tool_call"
        }
      ],
      "additional_kwargs": {},
      "response_metadata": {},
      "invalid_tool_calls": []
    },
    {
      "id": "3c32ce72-338f-4217-aa35-57ba020e2aa3",
      "name": "analyze_seller_reliability",
      "type": "tool",
      "status": "success",
      "content": "Sim, o seller 3442f8959a84dea7ee197c632cb2df15 é considerado não confiável.\nMotivos:\n- Taxa de pedidos atrasados: 33.33% (acima de 5%)\n- Nota média de reviews: 3.0/5.0 (abaixo de 3.5)\n- Total de pedidos analisados: 3\n- Pedidos atrasados: 1",
      "tool_call_id": "seller_reliability_call_21",
      "additional_kwargs": {},
      "response_metadata": {}
    },
    {
      "id": "lc_run--7d833d01-f1fe-4097-b182-de350a02af22-0",
      "type": "ai",
      "content": "Não é confiável. Métricas:\n- Atrasos: 33,33% (acima de 5%)\n- Nota média: 3,0/5,0 (abaixo de 3,5)\n- Amostra: 3 pedidos (1 atrasado)",
      "tool_calls": [],
      "usage_metadata": {
        "input_tokens": 744,
        "total_tokens": 876,
        "output_tokens": 132,
        "input_token_details": {
          "audio": 0,
          "cache_read": 0
        },
        "output_token_details": {
          "audio": 0,
          "reasoning": 64
        }
      },
      "additional_kwargs": {
        "refusal": null
      },
      "response_metadata": {
        "id": "chatcmpl-CdGD8LskGYzBRJKGb0BkDgaCSfsET",
        "logprobs": null,
        "model_name": "gpt-5-2025-08-07",
        "token_usage": {
          "total_tokens": 876,
          "prompt_tokens": 744,
          "completion_tokens": 132,
          "prompt_tokens_details": {
            "audio_tokens": 0,
            "cached_tokens": 0
          },
          "completion_tokens_details": {
            "audio_tokens": 0,
            "reasoning_tokens": 64,
            "accepted_prediction_tokens": 0,
            "rejected_prediction_tokens": 0
          }
        },
        "service_tier": "default",
        "finish_reason": "stop",
        "model_provider": "openai",
        "system_fingerprint": null
      },
      "invalid_tool_calls": []
    },
    {
      "id": "350c9af6-d946-490c-94c7-26cc536a553a",
      "type": "human",
      "content": "Processe a devolução do pedido e481f51cbdc54678b7cc49136f2d6af7",
      "additional_kwargs": {},
      "response_metadata": {}
    },
    {
      "id": "627240a3-5099-4fda-b49b-232bec0d590a",
      "type": "ai",
      "content": "Pedido e481f51cbdc54678b7cc49136f2d6af7 foi marcado como devolvido (return_processed) com sucesso.",
      "tool_calls": [],
      "additional_kwargs": {},
      "response_metadata": {},
      "invalid_tool_calls": []
    },
    {
      "id": "lc_run--d039b807-1681-44b3-b083-2054f084558d-0",
      "type": "ai",
      "content": "Devolução do pedido e481f51cbdc54678b7cc49136f2d6af7 processada. Status atualizado para return_processed.",
      "tool_calls": [],
      "usage_metadata": {
        "input_tokens": 639,
        "total_tokens": 874,
        "output_tokens": 235,
        "input_token_details": {
          "audio": 0,
          "cache_read": 0
        },
        "output_token_details": {
          "audio": 0,
          "reasoning": 192
        }
      },
      "additional_kwargs": {
        "refusal": null
      },
      "response_metadata": {
        "id": "chatcmpl-CdGGGLcD4PAxd9pflu5SHJSoHeq37",
        "logprobs": null,
        "model_name": "gpt-5-2025-08-07",
        "token_usage": {
          "total_tokens": 874,
          "prompt_tokens": 639,
          "completion_tokens": 235,
          "prompt_tokens_details": {
            "audio_tokens": 0,
            "cached_tokens": 0
          },
          "completion_tokens_details": {
            "audio_tokens": 0,
            "reasoning_tokens": 192,
            "accepted_prediction_tokens": 0,
            "rejected_prediction_tokens": 0
          }
        },
        "service_tier": "default",
        "finish_reason": "stop",
        "model_provider": "openai",
        "system_fingerprint": null
      },
      "invalid_tool_calls": []
    }
  ],
  "decide_path": "process_return",
  "pdf_context": ""
}

In [21]:
results = evaluate(
    run_agent,
    data="agent-final-output",
    evaluators=[relevance],
)

View the evaluation results for experiment: 'terrific-son-4' at:
https://smith.langchain.com/o/1a2f48d3-e49c-4ebd-b03e-aeaedc034215/datasets/cb9bb7e9-467e-48eb-ab7e-1dfa1ff17cac/compare?selectedSessions=1c2f5070-4359-4bb6-baa6-c4947f7c8b97




0it [00:00, ?it/s]

decide_path tool
decision: process_return
process_return_node: calling tool directly
Generating final answer...


1it [00:09,  9.71s/it]

decide_path tool
decision: sql_branch
call_get_schema: getting schema directly
generate_query tool
should_continue
Generating final answer...


...

5it [01:19, 17.36s/it]

decide_path tool
decision: analyze_seller_reliability
analyze_seller_reliability_node_custom: calling tool directly
Generating final answer...


6it [01:30, 15.46s/it]

decide_path tool
decision: process_return
process_return_node: calling tool directly
Generating final answer...


7it [01:44, 14.88s/it]


# Branch Correctness

In [ ]:
from langsmith import Client
ls_client = Client()

examples = [
  {
    "inputs": {"question": "Há quantos pedidos com status cancelado?"},
    "outputs": {"branch": "sql_branch"},
  },
  {
    "inputs": {"question": "Quais 3 vendedores tiveram o maior número de entregas atrasadas em 2024?"},
    "outputs": {"branch": "sql_branch"},
  },
  {
    "inputs": {"question": "No caso de devolução quem arca com custo do envio?"},
    "outputs": {"branch": "pdf_branch"},
  },
  {
    "inputs": {"question": "Qual o prazo de devolução por defeito da BIX?"},
    "outputs": {"branch": "pdf_branch"},
  },
  {
    "inputs": {"question": "O pedido dd787ad9c97e5504d6ea0bd294906902 está dentro do prazo de devolução por arrependimento?"},
    "outputs": {"branch": "pdf_sql_branch"},
  },
  {
    "inputs": {"question": "Quantos pedidos entregues da base de dados estão dentro do prazo de devolução por defeito? Considere que são todos itens não duráveis"},
    "outputs": {"branch": "pdf_sql_branch"},
  },
  {
    "inputs": {"question": "O seller 3442f8959a84dea7ee197c632cb2df15 é confiável?"},
    "outputs": {"branch": "analyze_seller_reliability"},
  },
  {
    "inputs": {"question": "Quais os top 3 vendedores menos confiáveis de 2025?"},
    "outputs": {"branch": "analyze_seller_reliability"},
  },
  {
    "inputs": {"question": "Processe devolução do pedido 2591f6277be80b0c25627c745ec900c4"},
    "outputs": {"branch": "process_return"},
  },
]
dataset = ls_client.create_dataset(dataset_name="branch-decision")
ls_client.create_examples(
  dataset_id=dataset.id,
  examples=examples,
)

In [91]:
def run_agent(inputs: dict):
    # Extract the user message
    user_msg = inputs.get("question", [])

    thread_id = f"eval-{uuid.uuid4()}"

    # Agent invocation
    result = agent.invoke(
        {"messages": [HumanMessage(content=user_msg)]},
        config={"thread_id": thread_id}
    )

    # Return final answer and branch decision
    branch_decision = result.get("decide_path", "unknown")

    return {"branch": branch_decision}


In [ ]:
def branch_correctness(outputs: dict, reference_outputs: dict) -> dict:
    """
    Simple exact match evaluator for branch correctness.
    Checks if the predicted branch exactly matches the expected branch.
    Returns a score of 1.0 if match, 0.0 otherwise.
    """
    predicted_branch = outputs.get("branch", "")
    expected_branch = reference_outputs.get("branch", "")
    
    # Exact match check
    is_correct = predicted_branch == expected_branch
    
    return {
        "key": "branch_correctness",
        "score": 1.0 if is_correct else 0.0
    }


In [95]:
results = evaluate(
    run_agent,
    data="branch-decision",
    evaluators=[branch_correctness],
)

View the evaluation results for experiment: 'healthy-glove-42' at:
https://smith.langchain.com/o/1a2f48d3-e49c-4ebd-b03e-aeaedc034215/datasets/375cfe16-5c1f-4b73-a81c-1eeb21b6a693/compare?selectedSessions=f3991f91-f671-4f52-9e0c-f6d7d8f274cf




0it [00:00, ?it/s]

decide_path tool
decision: analyze_seller_reliability
analyze_seller_reliability_node_custom: calling tool directly
Generating final answer...


1it [00:14, 14.51s/it]

decide_path tool
decision: pdf_branch
Running PDF branch...
pdf_context loaded: 3710 characters
Generating final answer...


...

7it [01:46, 15.96s/it]

decide_path tool
decision: sql_branch
call_get_schema: getting schema directly
generate_query tool
should_continue
Generating final answer...


8it [02:15, 20.02s/it]

decide_path tool
decision: pdf_sql_branch
Running PDF branch...
pdf_context loaded: 3710 characters
call_get_schema: getting schema directly
generate_query tool
should_continue
Generating final answer...


9it [02:40, 17.85s/it]


# Computation Accuracy

In [17]:
from langsmith import Client
ls_client = Client()

examples = [
  {
    "inputs": {"question": "Há quantos pedidos com status cancelado?"},
    "outputs": {"numeric_answer": "625"},
  },
  {
    "inputs": {"question": "Quantos pedidos têm método de pagamento cartão de crédito?"},
    "outputs": {"numeric_answer": "76505"},
  },
  {
    "inputs": {"question": "Há quantos clientes com pelo menos um pedido?"},
    "outputs": {"numeric_answer": "98816"},
  },
  {
    "inputs": {"question": "Quantos produtos têm avaliação abaixo de 3 estrelas?"},
    "outputs": {"numeric_answer": "4421"},
  },
  {
    "inputs": {"question": "Quantos pedidos estão em trânsito?"},
    "outputs": {"numeric_answer": "1107"},
  },
]
dataset = ls_client.create_dataset(dataset_name="numeric-answers")
ls_client.create_examples(
  dataset_id=dataset.id,
  examples=examples,
)

{'example_ids': ['63f5f88a-eb92-4caa-9436-67059c49f803',
  '1df398e1-7db3-46c3-ae83-e97948170622',
  'c2e40ed9-f9e6-47d1-8163-8bfc0950e44b',
  'e0e9b879-a6c7-4cae-8d8b-310ac7b0ec84',
  '1b0febfd-1f0b-45c1-9672-005283de5bd7'],
 'count': 5}

In [18]:
def run_agent(inputs: dict):
    # Extract the user message
    user_msg = inputs.get("question", "")

    thread_id = f"eval-{uuid.uuid4()}"

    # Agent invocation
    result = agent.invoke(
        {"messages": [HumanMessage(content=user_msg)]},
        config={"thread_id": thread_id}
    )

    # Extract the final answer from the agent's response
    final_answer = result["messages"][-1].content

    return {"prediction": final_answer}


In [30]:
import re

def extract_first_number(text: str):
    """
    Extracts the first number (int or decimal) from a text.
    Handles '.' or ',' as decimal separators.
    Returns float or None.
    """

    match = re.search(r"\d+[.,]?\d*", text)
    if not match:
        return None

    number_str = match.group(0)

    # Convert comma decimal to dot
    number_str = number_str.replace(",", ".")
    number_str = number_str.replace(".", "")

    return number_str


In [32]:
def computation_accuracy(outputs: dict, reference_outputs: dict) -> dict:
    """
    Simple exact match evaluator for computation accuracy.
    Extracts numeric value from the prediction and compares with expected numeric answer.
    Returns a score of 1.0 if match, 0.0 otherwise.
    """
    import re
    
    # Extract numeric answer from agent's prediction
    prediction = outputs.get("prediction", "")
    
    # This handles cases like "Há 4.421 pedidos" -> extracts "4421"
    predicted_number = extract_first_number(prediction)
    
    # Get expected numeric answer from reference
    expected_number = reference_outputs.get("numeric_answer", "")
    
    # Exact match check (both as strings for exact comparison)
    is_correct = predicted_number == expected_number
    
    return {
        "key": "computation_accuracy",
        "score": 1.0 if is_correct else 0.0
    }


In [33]:
results = evaluate(
    run_agent,
    data="numeric-answers",
    evaluators=[computation_accuracy],
)

View the evaluation results for experiment: 'shiny-page-73' at:
https://smith.langchain.com/o/1a2f48d3-e49c-4ebd-b03e-aeaedc034215/datasets/1b713cc6-d6cd-482f-9ef3-b3313ee84f95/compare?selectedSessions=12d7a429-7d47-442f-87c3-e5d6bcbd8c9a




0it [00:00, ?it/s]

decide_path tool
decision: sql_branch
call_get_schema: getting schema directly
generate_query tool
should_continue
Generating final answer...


1it [00:13, 14.00s/it]

decide_path tool
decision: sql_branch
call_get_schema: getting schema directly
generate_query tool
should_continue
Generating final answer...


...

3it [00:37, 12.34s/it]

decide_path tool
decision: sql_branch
call_get_schema: getting schema directly
generate_query tool
should_continue
Generating final answer...


4it [00:51, 13.13s/it]

decide_path tool
decision: sql_branch
call_get_schema: getting schema directly
generate_query tool
should_continue
Generating final answer...


5it [01:16, 15.25s/it]


# Relevance + Branch Correctness + Hallucination + Accuracy

In [74]:
def prepare_data(run, example):
    # USER QUESTION (from example input)
    user_msg = example.inputs["messages"][0]["content"]

    # REFERENCE ANSWER (from example output)
    ai_messages = [m for m in example.outputs["messages"] if m["type"] == "ai"]
    reference = ai_messages[-1]["content"] if ai_messages else ""

    # MODEL ANSWER (from run)
    output = run.outputs.get("prediction", "")

    return {
        "question": user_msg,
        "reference": reference,
        "answer": output
    }



In [75]:
# Use an LLM-as-a-judge
oai_client = wrappers.wrap_openai(OpenAI())

In [76]:
def hallucination(run: Run, example: Example) -> dict:
    """
    Avalia HALLUCINAÇÃO da resposta.
    Retorna nota 0..1.
    """
    instructions = """
    Você é um avaliador especializado em identificar HALLUCINAÇÕES em respostas de modelos.

    IMPORTANTE:
    - NÃO exija que a resposta repita literalmente a pergunta.
    - NÃO penalize respostas que adicionam detalhes verdadeiros, comuns ou plausíveis dentro do domínio.
    - Penalize APENAS informações inventadas, impossíveis, contraditórias ou não condizentes com práticas reais.

    CRITÉRIOS (nota 0.0 a 1.0):
    - 1.0 → Resposta totalmente plausível e consistente com o domínio. Nenhuma informação inventada.
    - 0.5 → Há detalhes especulativos, mas a intenção geral é plausível.
    - 0.0 → Afirmações falsas, inventadas, contraditórias ou absurdas.

    INSTRUÇÕES:
    - Avalie HALLUCINAÇÃO, não completude.
    - Verifique se a resposta é compatível com práticas normais do cenário descrito.
    - Retorne APENAS um número entre 0 e 1, nada mais.
    """


    data = prepare_data(run, example)

    msg = (
        f"Question: {data['question']}\n"
        f"Answer: {data['answer']}"
    )

    resp = oai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": msg}
        ],
    )

    score_str = resp.choices[0].message.content.strip()
    score = float(score_str)

    return {"key": "hallucination", "score": score}


In [77]:
def relevance(run: Run, example: Example) -> dict:
    """
    Avalia CLAREZA + RELEVÂNCIA da resposta.
    Retorna nota 0..1.
    """
    instructions = """
    Avalie a resposta com base em CLAREZA e RELEVÂNCIA (nota 0 a 1).
    - 1.0 = resposta completa, clara e diretamente relacionada.
    - 0.5 = resposta parcialmente correta.
    - 0.0 = resposta incorreta ou fora do contexto.
    Retorne APENAS a nota, nada mais.
    """

    data = prepare_data(run, example)

    msg = (
        f"Question: {data['question']}\n"
        f"Answer: {data['answer']}"
    )

    resp = oai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": msg}
        ],
    )

    score_str = resp.choices[0].message.content.strip()
    score = float(score_str)

    return {"key": "relevance", "score": score}


In [78]:
def accuracy(run: Run, example: Example) -> dict:
    """
    Avalia ACURÁCIA da resposta comparando com a referência.
    Retorna nota 0..1.
    """

    data = prepare_data(run, example)

    instructions = """
    Você é um avaliador especializado em verificar ACURÁCIA de respostas.

    Avalie o quão correta é a resposta do modelo comparada com a RESPOSTA DE REFERÊNCIA.

    CRITÉRIOS (nota de 0.0 a 1.0):
    - 1.0 → A resposta está totalmente correta, sem diferenças factuais relevantes.
    - 0.5 → Parcialmente correta: contém partes verdadeiras, mas erra ou omite detalhes importantes.
    - 0.0 → Incorreta: contradiz a referência ou apresenta informações erradas.

    INSTRUÇÕES:
    - Compare apenas FATO e CONTEÚDO, sem considerar estilo ou redação.
    - Use bom senso para equivalências (“dez” = “10”, sinônimos, ordem invertida, etc.).
    - Retorne APENAS um número entre 0 e 1.
    """

    msg = (
        f"Question: {data['question']}\n"
        f"Reference Answer: {data['reference']}\n"
        f"Model Answer: {data['answer']}"
    )

    resp = oai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": msg}
        ],
    )

    raw = resp.choices[0].message.content.strip()

    # Convert to float safely
    try:
        score = float(raw)
    except:
        import re
        m = re.search(r"\d+(\.\d+)?", raw)
        score = float(m.group(0)) if m else 0.0

    # Clamp 0–1 to prevent model mistakes
    score = max(0.0, min(1.0, score))

    return {"key": "accuracy", "score": score}


In [79]:
# Modified run_agent for agent-final-output dataset that returns both prediction and branch
def run_agent_combined(inputs: dict):
    """
    Modified run_agent that returns both prediction and branch decision.
    Works with agent-final-output dataset format (uses 'messages' input).
    Based on input_example and output_example structure.
    """
    # Extract the user message from inputs (agent-final-output format)
    messages = inputs.get("messages", [])
    if not messages:
        raise ValueError("Dataset missing 'messages' list.")

    user_msg = messages[0]["content"]

    thread_id = f"eval-{uuid.uuid4()}"

    # Agent invocation
    result = agent.invoke(
        {"messages": [HumanMessage(content=user_msg)]},
        config={"thread_id": thread_id}
    )

    # Return both final answer and branch decision
    final_answer = result["messages"][-1].content
    branch_decision = result.get("decide_path", "unknown")

    return {"prediction": final_answer, "branch": branch_decision}


In [80]:
# Modified branch_correctness evaluator that works with Run/Example signature
# This allows it to work alongside the relevance evaluator
def branch_correctness(run: Run, example: Example) -> dict:
    """
    Branch correctness evaluator using Run/Example signature.
    Extracts branch from run outputs and compares with example outputs.
    Works with agent-final-output dataset format where decide_path is in the state.
    Returns a score of 1.0 if match, 0.0 otherwise.
    """
    # Extract predicted branch from run outputs
    predicted_branch = run.outputs.get("branch", "")
    
    # Extract expected branch from example outputs
    # For agent-final-output dataset, decide_path is in the output state
    expected_branch = ""
    if example.outputs:
        # Try to get branch from outputs directly (for branch-decision dataset)
        expected_branch = example.outputs.get("branch", "")
        # If not found, try to extract from decide_path in the state (for agent-final-output)
        if not expected_branch:
            expected_branch = example.outputs.get("decide_path", "")
    
    # Exact match check
    is_correct = predicted_branch == expected_branch
    
    return {
        "key": "branch_correctness",
        "score": 1.0 if is_correct else 0.0
    }


In [81]:
# Run evaluation on agent-final-output dataset with both evaluators
# This evaluates both relevance (LLM-based) and branch correctness (exact match)
results_combined = evaluate(
    run_agent_combined,
    data="agent-final-output",
    evaluators=[relevance, branch_correctness, hallucination, accuracy],
    experiment_prefix="combined-relevance-branch-hallucination",
    description="Combined evaluation of relevance, branch correctness and hallucination on agent-final-output dataset",
)


View the evaluation results for experiment: 'combined-relevance-branch-hallucination-610839ac' at:
https://smith.langchain.com/o/1a2f48d3-e49c-4ebd-b03e-aeaedc034215/datasets/cb9bb7e9-467e-48eb-ab7e-1dfa1ff17cac/compare?selectedSessions=cb6d0768-aea3-4683-8d2b-bd7353e70ed5




0it [00:00, ?it/s]

decide_path tool
decision: process_return
process_return_node: calling tool directly
Generating final answer...


1it [00:13, 13.26s/it]

decide_path tool
decision: sql_branch
call_get_schema: getting schema directly
generate_query tool
should_continue
Generating final answer...


...

5it [01:30, 19.95s/it]

decide_path tool
decision: analyze_seller_reliability
analyze_seller_reliability_node_custom: calling tool directly
Generating final answer...


6it [01:46, 18.63s/it]

decide_path tool
decision: process_return
process_return_node: calling tool directly
Generating final answer...


7it [02:02, 17.49s/it]
